## PLUTO – Test Problems

In this notebook, we verify that the code installation works as intended and get familiar with the analysis of PLUTO simulation results.

### 0. Preamble

In [1]:
%%capture

%config InlineBackend.figure_formats = ['retina']

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

import pyPLUTO as pp

import io
import base64

from PIL import Image
from IPython.display import Image as HTML, display
import IPython.display as ipd

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import Normalize, LogNorm
from matplotlib.ticker import ScalarFormatter, MultipleLocator
from matplotlib.patches import Wedge

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=["steelblue", "olivedrab", "goldenrod", "firebrick", "rebeccapurple"]) 
plt.rcParams['figure.figsize'] = [8,5]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["ytick.minor.visible"] = True

plt.plot()
plt.close()

### 1. Simple Shock Tube

A standard hydrodynamics benchmark problem in one dimension with open boundaries, where density and pressure have a sharp discontinuity in their initial conditions:

$$\begin{align*}
\begin{pmatrix}\rho \\ p \\ v\end{pmatrix} = \begin{pmatrix}1 \\ 1 \\ 0\end{pmatrix} && x \leq 0.5 &&&&&&&&
\begin{pmatrix}\rho \\ p \\ v\end{pmatrix} = \begin{pmatrix}1/8 \\ 1/10 \\ 0\end{pmatrix} && x > 0.5
\end{align*}$$

The panels below exhibit a rarefaction wave moving leftward, as well as a discontinuity and shock wave propagating to the right. Default settings were used for the simulator setup step.

In [2]:
path = 'HD/Sod/'
D_last = pp.Load(nout='last', path=path)

outlist = list(D_last.outlist)
outlist = [outlist[0]] * 3 + outlist + [outlist[-1]] * 3

frames = []

for i in outlist:
    D = pp.Load(nout=i, path=path)
    fig, axs = plt.subplots(3, 1, figsize=[6, 12], sharex=True)

    axs[0].plot(D.x1, D.rho, 'k-', ms=3, lw=1.5)
    axs[0].set_ylabel(r'$\rho$')
    axs[0].set_ylim(0, 1.1)
    axs[0].text(1.0125, 0.5, 'Density', rotation=-90, va='center', ha='left', transform=axs[0].transAxes)

    axs[1].plot(D.x1, D.prs, 'k-', ms=3, lw=1.5)
    axs[1].set_ylabel(r'$p$')
    axs[1].set_ylim(0, 1.1)
    axs[1].text(1.0125, 0.5, 'Pressure', rotation=-90, va='center', ha='left', transform=axs[1].transAxes)

    axs[2].plot(D.x1, D.vx1, 'k-', ms=3, lw=1.5)
    axs[2].set_ylabel(r'$v$')
    axs[2].set_ylim(-0.1, 1.1)
    axs[2].text(1.0125, 0.5, 'Velocity', rotation=-90, va='center', ha='left', transform=axs[2].transAxes)

    axs[-1].set_xlabel(r'$x$')

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=200)
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    frames.append(Image.open(buf).convert('RGB'))

gif_path = f'{path}/Storage/panel.gif'
frames[0].save(gif_path, format='GIF', save_all=True, append_images=frames[1:], duration=150, loop=0)
ipd.display(ipd.HTML(f'<img src="{gif_path}" width="600">'))

### 2. Orszag Tang Vortex

A simple and reproducible system with a double periodic fluid configuration in two dimensions. Initial conditions are set as follows:

$$\begin{align*}
v = \begin{pmatrix}-\sin y \\ \sin x \end{pmatrix} &&&&
B = \begin{pmatrix}-\sin y \\ \sin 2x \end{pmatrix} &&&&
\rho = 25/9 &&&& p = 5/3
\end{align*}$$

This setup leads to the formation of current sheets and supersonic magnetohydrodynamical turbulence.

#### 2.1 Density

In [3]:
path = 'MHD/Orszag_Tang/'
D_last = pp.Load(nout='last', path=path)
unique_outs = list(D_last.outlist)

rho_frames = {}
for i in unique_outs:
    Di = pp.Load(nout=i, path=path)
    rho_frames[i] = Di.rho.copy()

vmin = min(r.min() for r in rho_frames.values())
vmax = max(r.max() for r in rho_frames.values())

outlist = [unique_outs[0]] * 3 + unique_outs + [unique_outs[-1]] * 3

frames = []
for i in outlist:
    D = pp.Load(nout=i, path=path)

    fig, ax = plt.subplots(figsize=[6, 6])
    im = ax.pcolormesh(D.x1, D.x2, rho_frames[i].T, cmap='plasma', norm=Normalize(vmin=vmin, vmax=vmax), shading='auto')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_aspect('equal')

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    fig.colorbar(im, cax=cax, label=r'$\rho$')

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    frames.append(img.convert('RGB'))

gif_path = f'{path}/Storage/density.gif'
frames[0].save(gif_path, format='GIF', save_all=True, append_images=frames[1:], duration=150, loop=0)
ipd.display(ipd.HTML(f'<img src="{gif_path}" width="600">'))

#### 2.2 Velocity Magnitude

$$|v| = \sqrt{v_x^2 + v_y^2}$$

In [4]:
path = 'MHD/Orszag_Tang/'
D_last = pp.Load(nout='last', path=path)
unique_outs = list(D_last.outlist)

vmag_frames = {}
for i in unique_outs:
    Di = pp.Load(nout=i, path=path)
    vmag_frames[i] = np.sqrt(Di.vx1**2 + Di.vx2**2)

vmin = min(v.min() for v in vmag_frames.values())
vmax = max(v.max() for v in vmag_frames.values())

outlist = [unique_outs[0]] * 3 + unique_outs + [unique_outs[-1]] * 3

frames = []
for i in outlist:
    D = pp.Load(nout=i, path=path)
    fig, ax = plt.subplots(figsize=[6, 6])
    im = ax.pcolormesh(D.x1, D.x2, vmag_frames[i].T, cmap='plasma', norm=Normalize(vmin=vmin, vmax=vmax), shading='auto')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_aspect('equal')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    fig.colorbar(im, cax=cax, label=r'$|v|$')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    frames.append(img.convert('RGB'))

gif_path = f'{path}/Storage/velocity.gif'
frames[0].save(gif_path, format='GIF', save_all=True, append_images=frames[1:], duration=150, loop=0)
ipd.display(ipd.HTML(f'<img src="{gif_path}" width="600">'))

#### 2.3 Magnetic Field Magnitude

$$|B| = \sqrt{B_x^2 + B_y^2}$$

In [5]:
path = 'MHD/Orszag_Tang/'
D_last = pp.Load(nout='last', path=path)
unique_outs = list(D_last.outlist)

bmag_frames = {}
for i in unique_outs:
    Di = pp.Load(nout=i, path=path)
    bmag_frames[i] = np.sqrt(Di.Bx1**2 + Di.Bx2**2)

vmin = min(b.min() for b in bmag_frames.values())
vmax = max(b.max() for b in bmag_frames.values())

outlist = [unique_outs[0]] * 3 + unique_outs + [unique_outs[-1]] * 3

frames = []
for i in outlist:
    D = pp.Load(nout=i, path=path)
    fig, ax = plt.subplots(figsize=[6, 6])
    im = ax.pcolormesh(D.x1, D.x2, bmag_frames[i].T, cmap='plasma', norm=Normalize(vmin=vmin, vmax=vmax), shading='auto')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_aspect('equal')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    fig.colorbar(im, cax=cax, label=r'$|B|$')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    frames.append(img.convert('RGB'))

gif_path = f'{path}/Storage/field.gif'
frames[0].save(gif_path, format='GIF', save_all=True, append_images=frames[1:], duration=150, loop=0)
ipd.display(ipd.HTML(f'<img src="{gif_path}" width="600">'))

#### 2.4 Current Density

$$J_z = \partial_x B_y - \partial_y B_x$$

In [6]:
path = 'MHD/Orszag_Tang/'
D_last = pp.Load(nout='last', path=path)
unique_outs = list(D_last.outlist)

jz_frames = {}
for i in unique_outs:
    Di = pp.Load(nout=i, path=path)
    dBy_dx = np.gradient(Di.Bx2, Di.x1, axis=0)
    dBx_dy = np.gradient(Di.Bx1, Di.x2, axis=1)
    jz_frames[i] = dBy_dx - dBx_dy

absmax = max(np.abs(j).max() for j in jz_frames.values())
vmin, vmax = -absmax, absmax

outlist = [unique_outs[0]] * 3 + unique_outs + [unique_outs[-1]] * 3

frames = []
for i in outlist:
    D = pp.Load(nout=i, path=path)
    fig, ax = plt.subplots(figsize=[6, 6])
    im = ax.pcolormesh(D.x1, D.x2, jz_frames[i].T, cmap='RdBu_r', norm=Normalize(vmin=vmin, vmax=vmax), shading='auto')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_aspect('equal')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    fig.colorbar(im, cax=cax, label=r'$J_z$', extend='neither')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    frames.append(img.convert('RGB'))

gif_path = f'{path}/Storage/current.gif'
frames[0].save(gif_path, format='GIF', save_all=True, append_images=frames[1:], duration=150, loop=0)
ipd.display(ipd.HTML(f'<img src="{gif_path}" width="600">'))

### 3. Accretion Disks

In [3]:
def compute_flux_function(D):
    Br = D.Bx1
    theta = D.x2
    r = D.x1
    integrand = Br * (r[:, None]**2) * np.sin(theta)[None, :]
    Psi = np.zeros_like(integrand)
    dtheta = np.diff(theta)
    Psi[:, 1:] = np.cumsum(
        0.5 * (integrand[:, 1:] + integrand[:, :-1]) * dtheta[None, :],
        axis=1
    )
    return Psi



def animate_density(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    
    rho_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
    
    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])
    
    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20
    
    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
    
        im = ax.pcolormesh(
            X, Z, rho_frames[i], 
            cmap='magma', 
            norm=LogNorm(vmin=vmin, vmax=vmax), 
            shading='auto'
        )
    
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
    
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')
    
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')
    
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/density.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_velocity(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist
    
    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    
    rho_frames = {}
    vx_frames = {}
    vz_frames = {}
    vr_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
    
        vr = Di.vx1
        vth = Di.vx2
        vx = vr * np.sin(Theta) + vth * np.cos(Theta)
        vz = vr * np.cos(Theta) - vth * np.sin(Theta)
    
        vx_frames[i] = np.where(mask, vx, np.nan)
        vz_frames[i] = np.where(mask, vz, np.nan)
        vr_frames[i] = np.where(mask, vr, np.nan)
    
    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])
    
    stride1, stride2 = 1, 1
    
    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        im = ax.pcolormesh(
            X, Z, rho_frames[i],
            cmap='gray',
            norm=LogNorm(vmin=vmin, vmax=vmax),
            shading='auto'
        )
        Xq = X[::stride1, ::stride2]
        Zq = Z[::stride1, ::stride2]
        Uq = vx_frames[i][::stride1, ::stride2]
        Wq = vz_frames[i][::stride1, ::stride2]
        Vrq = vr_frames[i][::stride1, ::stride2]
    
        colors = np.where(Vrq >= 0, 'r', 'b').ravel()
    
        ax.quiver(
            Xq, Zq, Uq, Wq,
            color=colors,
            scale_units='xy',
            angles='xy',
        )
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/velocity.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_field(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)

    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)

    rho_frames = {}
    Psi_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
        Psi_frames[i] = compute_flux_function(Di)

    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])

    global_absmax = max(np.nanmax(np.abs(p)) for p in Psi_frames.values())
    levels_pos = np.linspace(0, global_absmax, 100)
    levels_neg = -levels_pos[::-1]

    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')

        im = ax.pcolormesh(
            X, Z, rho_frames[i],
            cmap='gray',
            norm=LogNorm(vmin=vmin, vmax=vmax),
            shading='auto'
        )

        ax.contour(X, Z, Psi_frames[i], levels=levels_pos, colors='m', linestyle='-', linewidths=0.4)
        ax.contour(X, Z, Psi_frames[i], levels=levels_neg, colors='m', linestyle='-', linewidths=0.4)

        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    gif_path = f'{path}/Storage/field.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )

    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_panel(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    def profiles(D):
        rho = D.rho
        prs = D.prs
        vr = D.vx1
        vphi = D.vx3
        tr = D.tr1
        theta = D.x2
        r = D.x1

        sin_th = np.sin(theta)[None, :]
        l_specific = r[:, None] * np.sin(theta)[None, :] * vphi

        def weighted_avg(field, weight):
            num = np.trapezoid(field * weight * sin_th, theta, axis=1)
            den = np.trapezoid(weight * sin_th, theta, axis=1)
            return num / np.where(den == 0, np.nan, den)

        rho_disk = weighted_avg(rho, tr)
        rho_corona = weighted_avg(rho, 1 - tr)
        prs_disk = weighted_avg(prs, tr)
        prs_corona = weighted_avg(prs, 1 - tr)

        base = rho * vr * sin_th
        base_l = rho * vr * l_specific * sin_th
        Mdot_disk = 2 * np.pi * r**2 * np.trapezoid(base * tr, theta, axis=1)
        Mdot_corona = 2 * np.pi * r**2 * np.trapezoid(base * (1 - tr), theta, axis=1)
        Ldot_disk = 2 * np.pi * r**2 * np.trapezoid(base_l * tr, theta, axis=1)
        Ldot_corona = 2 * np.pi * r**2 * np.trapezoid(base_l * (1 - tr), theta, axis=1)

        return (r, rho_disk, rho_corona, prs_disk, prs_corona,
                Mdot_disk, Mdot_corona, Ldot_disk, Ldot_corona)

    r_ref = None
    rho_disk_frames, rho_corona_frames = {}, {}
    prs_disk_frames, prs_corona_frames = {}, {}
    Mdot_disk_frames, Mdot_corona_frames = {}, {}
    Ldot_disk_frames, Ldot_corona_frames = {}, {}

    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        r, rd, rc, pd, pc, md, mc, ld, lc = profiles(Di)
        r_ref = r
        rho_disk_frames[i] = rd
        rho_corona_frames[i] = rc
        prs_disk_frames[i] = pd
        prs_corona_frames[i] = pc
        Mdot_disk_frames[i] = md
        Mdot_corona_frames[i] = mc
        Ldot_disk_frames[i] = ld
        Ldot_corona_frames[i] = lc

    def shared_ylim(*frame_dicts, log=False):
        all_vals = np.concatenate([v[~np.isnan(v)] for d in frame_dicts for v in d.values()])
        if log:
            all_vals = all_vals[all_vals > 0]
            ymin, ymax = all_vals.min(), all_vals.max()
            pad = (ymax / ymin) ** 0.05
            return ymin / pad, ymax * pad
        ymin, ymax = all_vals.min(), all_vals.max()
        pad = 0.05 * (ymax - ymin)
        return ymin - pad, ymax + pad

    rho_ymin, rho_ymax = shared_ylim(rho_disk_frames, rho_corona_frames, log=True)
    prs_ymin, prs_ymax = shared_ylim(prs_disk_frames, prs_corona_frames, log=True)
    mdot_ymin, mdot_ymax = shared_ylim(Mdot_disk_frames, Mdot_corona_frames)
    ldot_ymin, ldot_ymax = shared_ylim(Ldot_disk_frames, Ldot_corona_frames)

    outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10

    frames = []
    for i in outlist:
        fig, axs = plt.subplots(4, 2, figsize=[7.5, 10], sharex=True)

        for col, frame_dict in [(0, rho_disk_frames), (1, rho_corona_frames)]:
            ax = axs[0, col]
            y = frame_dict[i]
            ax.fill_between(r_ref, y, rho_ymin, color='m', alpha=1/4, linewidth=0)
            ax.set_yscale('log')
            ax.set_ylim(rho_ymin, rho_ymax)
            if col == 0:
                ax.set_ylabel(r'$\langle\rho\rangle$')

        for col, frame_dict in [(0, prs_disk_frames), (1, prs_corona_frames)]:
            ax = axs[1, col]
            y = frame_dict[i]
            ax.fill_between(r_ref, y, prs_ymin, color='m', alpha=1/4, linewidth=0)
            ax.set_yscale('log')
            ax.set_ylim(prs_ymin, prs_ymax)
            if col == 0:
                ax.set_ylabel(r'$\langle P\rangle$')

        panel_spec = [
            (2, 0, Mdot_disk_frames,   r'$\dot{M}$',    mdot_ymin, mdot_ymax),
            (2, 1, Mdot_corona_frames, None,            mdot_ymin, mdot_ymax),
            (3, 0, Ldot_disk_frames,   r'$\dot{L}$',    ldot_ymin, ldot_ymax),
            (3, 1, Ldot_corona_frames, None,            ldot_ymin, ldot_ymax),
        ]
        for row, col, frame_dict, ylabel, ymin, ymax in panel_spec:
            ax = axs[row, col]
            y = frame_dict[i]
            ax.axhline(0, color='k', lw=0.8, ls='-')
            ax.fill_between(r_ref, y, 0, where=(y < 0), color='b', alpha=1/3, linewidth=0)
            ax.fill_between(r_ref, y, 0, where=(y >= 0), color='r', alpha=1/3, linewidth=0)
            ax.set_ylabel(ylabel)
            ax.set_ylim(ymin, ymax)

        for row in range(4):
            for col in range(2):
                ax = axs[row, col]
                ax.set_xscale('log')
                ax.set_xticks([2, 4, 6, 8, 10, 20, 30])
                ax.xaxis.set_major_formatter(ScalarFormatter())
                ax.minorticks_off()
                ax.xaxis.set_minor_formatter(plt.NullFormatter())
                if row == 3:
                    ax.set_xlabel(r'$r$')

        axs[0, 0].set_title('Disk')
        axs[0, 1].set_title('Corona')
        for row in range(4):
            axs[row, 1].tick_params(labelleft=False)

        if 'STAR' in path:
            axs[0, 1].plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            axs[0, 1].plot([], [], ' ', label=f't = {times[i]:.0f}')
        axs[0, 1].legend(loc='upper right')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    gif_path = f'{path}/Storage/panel.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))

#### 3.1. Stellar Surface

##### 3.1.1. Viscous Fluid

In [11]:
path = 'Custom/STAR_VISC_HD/'
animate_density(path)
animate_velocity(path)
animate_panel(path)

##### 3.1.2. Viscous Magnetic Dipole

In [12]:
path = 'Custom/STAR_VISC_MHD/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

##### 3.1.3. Viscous & Resistive Magnetic Dipole

In [13]:
path = 'Custom/STAR_VISC_RES_MHD/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

#### 3.2. Schwarzschild Black Hole

##### 3.2.1. Classical Gravity

In [14]:
path = 'Custom/BH_VISC_RES_MHD_N/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

##### 3.2.2. Modified Gravity

In [15]:
path = 'Custom/BH_VISC_RES_MHD_PW'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

#### 3.3. Comparison

In [4]:
plt.rcParams['figure.constrained_layout.use'] = False

paths = {
           'Viscous HD':     'Custom/STAR_VISC_HD',
          'Viscous MHD':     'Custom/STAR_VISC_MHD',
        'Resistive MHD':     'Custom/STAR_VISC_RES_MHD',
            'Newtonian':     'Custom/BH_VISC_RES_MHD_N',
    'Paczyński & Wiita':     'Custom/BH_VISC_RES_MHD_PW',
}
col_order = list(paths.keys())

xlim, ylim = (0, 7), (0, 7)
stride1, stride2 = 1, 1

# ---------------------------------------------------------------------
# Load and cache all data
# ---------------------------------------------------------------------
grid_data = {}
for col_label, path in paths.items():
    is_star = 'STAR' in path
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    times = D_last.timelist / 62.8318 if is_star else D_last.timelist

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    mask = (X >= xlim[0]) & (X <= (xlim[1] + 0.5)) & (Z >= ylim[0]) & (Z <= (ylim[1] + 0.5))

    has_field = 'MHD' in path

    rho_frames, vx_frames, vz_frames, vr_frames = {}, {}, {}, {}
    Psi_frames = {} if has_field else None

    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)

        vr = Di.vx1
        vth = Di.vx2
        vx = vr * np.sin(Theta) + vth * np.cos(Theta)
        vz = vr * np.cos(Theta) - vth * np.sin(Theta)
        vx_frames[i] = np.where(mask, vx, np.nan)
        vz_frames[i] = np.where(mask, vz, np.nan)
        vr_frames[i] = np.where(mask, vr, np.nan)

        if has_field:
            Psi_frames[i] = compute_flux_function(Di)

    grid_data[col_label] = dict(
        X=X, Z=Z, rho_frames=rho_frames,
        vx_frames=vx_frames, vz_frames=vz_frames, vr_frames=vr_frames,
        Psi_frames=Psi_frames, has_field=has_field, is_star=is_star,
        unique_outs=unique_outs, times=times
    )

# Shared density scale across all rows/columns
vmin = np.nanmin([np.nanmin(r) for d in grid_data.values() for r in d['rho_frames'].values()])
vmax = np.nanmax([np.nanmax(r) for d in grid_data.values() for r in d['rho_frames'].values()])

# Shared field-line contour levels, only across columns that have a field
field_cols = [c for c in col_order if grid_data[c]['has_field']]
global_absmax = max(
    np.nanmax(np.abs(p)) for c in field_cols for p in grid_data[c]['Psi_frames'].values()
)
levels_pos = np.linspace(0, global_absmax, 500)[1::5]
levels_neg = -levels_pos[::-1]

n_frames = min(len(grid_data[c]['unique_outs']) for c in col_order)
outlist_idx = list(range(n_frames))
padded_idx = [outlist_idx[0]] * 10 + outlist_idx + [outlist_idx[-1]] * 10

# ---------------------------------------------------------------------
# Build frames
# ---------------------------------------------------------------------
frames = []
for k in padded_idx:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)

        fig, axs = plt.subplots(3, 5, figsize=[9.00, 5.25], sharex=True, sharey=True, layout=None)
        fig.subplots_adjust(left=0.05, right=0.88, top=0.94, bottom=0.09, wspace=0.06, hspace=0.05)

        for c_idx, col_label in enumerate(col_order):
            d = grid_data[col_label]
            path = paths[col_label]
            i = d['unique_outs'][k]

            # Row 0: density
            ax = axs[0, c_idx]
            ax.set_facecolor('k')
            im_density = ax.pcolormesh(
                d['X'], d['Z'], d['rho_frames'][i],
                cmap='magma', norm=LogNorm(vmin=vmin, vmax=vmax), shading='auto'
            )
            if 'STAR' in path:
                w = Wedge((0, 0), 1.0, 0, 90, facecolor='w', edgecolor=None)
            elif 'BH' in path:
                w = Wedge((0, 0), 2.1, 0, 90, facecolor='k', edgecolor=None)
            ax.add_patch(w)
            ax.set_aspect('equal')
            ax.set_title(col_label, fontsize=10)
            if c_idx == 0:
                ax.set_ylabel('z')

            # Row 1: velocity
            ax = axs[1, c_idx]
            ax.set_facecolor('k')
            ax.pcolormesh(
                d['X'], d['Z'], d['rho_frames'][i],
                cmap='gray', norm=LogNorm(vmin=vmin, vmax=vmax), shading='auto'
            )
            Xq = d['X'][::stride1, ::stride2]
            Zq = d['Z'][::stride1, ::stride2]
            Uq = d['vx_frames'][i][::stride1, ::stride2]
            Wq = d['vz_frames'][i][::stride1, ::stride2]
            Vrq = d['vr_frames'][i][::stride1, ::stride2]
            colors = np.where(Vrq >= 0, 'red', 'blue').ravel()

            q_mask = ~np.isnan(Uq.ravel()) & ~np.isnan(Wq.ravel())
            ax.quiver(
                Xq.ravel()[q_mask], Zq.ravel()[q_mask],
                Uq.ravel()[q_mask], Wq.ravel()[q_mask],
                color=colors[q_mask],
                scale_units='xy', angles='xy', scale=1.0
            )
            if 'STAR' in path:
                w = Wedge((0, 0), 1.0, 0, 90, facecolor='w', edgecolor=None)
            elif 'BH' in path:
                w = Wedge((0, 0), 2.1, 0, 90, facecolor='k', edgecolor=None)
            ax.add_patch(w)
            ax.set_aspect('equal')
            if c_idx == 0:
                ax.set_ylabel('z')

            # Row 2: magnetic field, or delete axis entirely
            if d['has_field']:
                ax = axs[2, c_idx]
                ax.set_facecolor('k')
                ax.pcolormesh(
                    d['X'], d['Z'], d['rho_frames'][i],
                    cmap='gray', norm=LogNorm(vmin=vmin, vmax=vmax), shading='auto'
                )
                ax.contour(d['X'], d['Z'], d['Psi_frames'][i], levels=levels_pos, colors='m', linewidths=0.4)
                ax.contour(d['X'], d['Z'], d['Psi_frames'][i], levels=levels_neg, colors='m', linewidths=0.4)
                if 'STAR' in path:
                    w = Wedge((0, 0), 1.0, 0, 90, facecolor='w', edgecolor=None)
                elif 'BH' in path:
                    w = Wedge((0, 0), 2.1, 0, 90, facecolor='k', edgecolor=None)
                ax.add_patch(w)
                ax.set_aspect('equal')
                ax.set_xlabel('R')
            else:
                fig.delaxes(axs[2, c_idx])
                axs[1, c_idx].set_xlabel('R')
                axs[1, c_idx].tick_params(labelbottom=True)
                axs[1, c_idx].set_ylabel('z', fontsize=12)

        axs[2, 1].set_ylabel('z')

        axs[0, 0].set_xlim(*xlim)
        axs[0, 0].set_ylim(*ylim)

        # Time labels: rotation periods for STAR columns, gravitational time units for BH columns
        star_cols = [c for c in col_order if grid_data[c]['is_star']]
        bh_cols = [c for c in col_order if not grid_data[c]['is_star']]

        if star_cols:
            ref_col = star_cols[-1]
            c_idx = col_order.index(ref_col)
            t_ref = grid_data[ref_col]['times'][grid_data[ref_col]['unique_outs'][k]]
            axs[0, c_idx].text(
                0.97, 0.97, f't = {t_ref:.1f}',
                transform=axs[0, c_idx].transAxes,
                ha='right', va='top', color='w', fontsize=10
            )

        if bh_cols:
            ref_col = bh_cols[-1]
            c_idx = col_order.index(ref_col)
            t_ref = grid_data[ref_col]['times'][grid_data[ref_col]['unique_outs'][k]]
            axs[0, c_idx].text(
                0.97, 0.97, f't = {t_ref:.0f}',
                transform=axs[0, c_idx].transAxes,
                ha='right', va='top', color='w', fontsize=10
            )

        cax = fig.add_axes([0.90, 0.09, 0.02, 0.85])
        fig.colorbar(im_density, cax=cax, label=r'$\rho$')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200)
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

gif_path = 'Custom/Storage/panel.gif'
frames[0].save(
    gif_path,
    format='GIF',
    save_all=True,
    append_images=frames[1:],
    duration=150,
    loop=0
)
ipd.display(ipd.HTML(f'<img src="{gif_path}" width="900">'))